# 🦺 EPP Detector — Inferencia con YOLO26

Detección de Equipos de Protección Personal (EPP) en obras de construcción.

**Requisitos previos:**
- Subí el archivo `best.pt` al entorno de Colab (panel izquierdo → 📁 → subir archivo)
- Activá GPU: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`

---

## VERIFICACIONES IMPORTACIONES Y CARGA DE MODELO

In [ ]:
!nvidia-smi

%pip install -q "ultralytics>=8.4.0" supervision

# Desactivar telemetría de Ultralytics
!yolo settings sync=False

import ultralytics
ultralytics.checks()

#CARGA DEL MODELO
import os
from ultralytics import YOLO

HOME = os.getcwd()

# ── URL directa del Release en GitHub ──────────────────────
MODEL_URL = "https://github.com/HoracioMantilla/MAIC1125-HMJ/releases/download/v1.0/best.pt"
MODEL_PATH = f"{HOME}/best.pt"
# ────────────────────────────────────────────────────────────

# Descargar solo si no existe ya en el entorno
if not os.path.exists(MODEL_PATH):
    print("Descargando modelo desde GitHub Releases...")
    !wget -q --show-progress -O {MODEL_PATH} {MODEL_URL}
    print("Descarga completada.")
else:
    print("Modelo ya presente en el entorno, omitiendo descarga.")

model = YOLO(MODEL_PATH)
print(f"\nModelo cargado correctamente.")
print(f"Clases: {model.names}")

# Definir función de anotación

import supervision as sv
from PIL import Image


def annotate(image: Image.Image, detections: sv.Detections) -> Image.Image:
    color = sv.ColorPalette.from_hex([
        "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00",
        "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    ])

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    box_annotator = sv.BoxAnnotator(color=color)
    label_annotator = sv.LabelAnnotator(
        color=color,
        text_color=sv.Color.BLACK,
        text_scale=text_scale,
        smart_position=True
    )

    out = image.copy()
    out = box_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections)
    out.thumbnail((1000, 1000))
    return out


print('Función annotate() lista.')



## Inferencia sobre una imagen propia

Subí una imagen desde tu computadora cuando se te solicite.

In [ ]:
from google.colab import files

print('Seleccioná una imagen para analizar...')
uploaded = files.upload()

image_name = list(uploaded.keys())[0]
image_path = f'{HOME}/{image_name}'

with open(image_path, 'wb') as f:
    f.write(uploaded[image_name])

# Inferencia
image = Image.open(image_path)
result = model.predict(image, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)
annotated_image = annotate(image, detections)

print(f'Detecciones encontradas: {len(detections)}')
annotated_image